In [ ]:
# Gridmatic ERCOT POC - Trading Forecast Analysis
# This notebook creates a trading-style forecast visualization

import os
import pandas as pd
from google.cloud import bigquery
from prophet import Prophet
import matplotlib.pyplot as plt

# Setup
PROJECT = os.environ["GCP_PROJECT"]
DATASET = os.environ.get("BQ_DATASET", "energy_ercot")
client = bigquery.Client(project=PROJECT)

print(f"🔍 Querying data from {PROJECT}.{DATASET}")
print("📊 Loading ERCOT demand data...")


In [ ]:
# Query BigQuery for ERCOT demand data
df = client.query(f"""
  SELECT timestamp_hour, demand_mw
  FROM `{PROJECT}.{DATASET}.silver_ercot_weather`
  WHERE timestamp_hour >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 365 DAY)
  ORDER BY timestamp_hour
""").to_dataframe()

print(f"✅ Loaded {len(df)} rows of demand data")
print(f"📅 Date range: {df['timestamp_hour'].min()} to {df['timestamp_hour'].max()}")
print(f"⚡ Demand range: {df['demand_mw'].min():.0f} - {df['demand_mw'].max():.0f} MW")


In [ ]:
# Train Prophet model and generate 48-hour forecast
print("🤖 Training Prophet model...")

# Prepare data for Prophet (remove timezone for compatibility)
dfp = df.rename(columns={"timestamp_hour":"ds", "demand_mw":"y"})
dfp['ds'] = dfp['ds'].dt.tz_localize(None)  # Remove timezone

# Train Prophet model
m = Prophet(daily_seasonality=True, weekly_seasonality=True)
m.fit(dfp)

# Generate 48-hour forecast
print("🔮 Generating 48-hour forecast...")
future = m.make_future_dataframe(periods=48, freq="H")
fc = m.predict(future)
fc_48 = fc.tail(48)

print(f"✅ Forecast complete: {len(fc_48)} hours ahead")
print(f"📈 Forecast range: {fc_48['yhat'].min():.0f} - {fc_48['yhat'].max():.0f} MW")


In [ ]:
# Write forecast to gold_forecasts table
print("💾 Writing forecast to BigQuery...")

forecast_output = fc_48[["ds","yhat","yhat_lower","yhat_upper"]].rename(columns={"ds":"timestamp_hour"})
client.load_table_from_dataframe(forecast_output, f"{PROJECT}.{DATASET}.gold_forecasts").result()

print("✅ Forecast saved to gold_forecasts table")
print(f"📊 Forecast summary:")
print(f"   - Start: {forecast_output['timestamp_hour'].min()}")
print(f"   - End: {forecast_output['timestamp_hour'].max()}")
print(f"   - Avg forecast: {forecast_output['yhat'].mean():.0f} MW")
print(f"   - Max forecast: {forecast_output['yhat'].max():.0f} MW")


In [ ]:
# PLOT #1 — Trading Graph (Main visualization)
plt.figure(figsize=(12,4))

# Get recent 7 days of actual data
recent = dfp.tail(7*24)

# Plot actual demand (last 7 days)
plt.plot(recent["ds"], recent["y"], 
         label="Actual (Demand)", 
         color='blue', 
         linewidth=1.5)

# Plot forecast (48 hours)
plt.plot(fc_48["ds"], fc_48["yhat"], 
         label="Forecast (48h)", 
         color='red', 
         linewidth=2)

# Add uncertainty band
plt.fill_between(fc_48["ds"], 
                 fc_48["yhat_lower"], 
                 fc_48["yhat_upper"], 
                 alpha=0.2, 
                 color='red',
                 label="Uncertainty")

# Formatting
plt.title("ERCOT Hourly Forecast vs Actual (Trading View)", fontsize=14, fontweight='bold')
plt.xlabel("Time")
plt.ylabel("Demand (MW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add vertical line to separate actual from forecast
if len(recent) > 0:
    last_actual = recent["ds"].iloc[-1]
    plt.axvline(x=last_actual, color='gray', linestyle='--', alpha=0.7)
    plt.text(last_actual, plt.ylim()[1]*0.95, 'Now', 
             rotation=90, ha='right', va='top', fontsize=10)

plt.show()

print("📈 Trading chart complete!")
print("💡 Note: When ERCOT DAM prices are available, replace 'demand_mw' with 'price' for price forecasting")


# ERCOT Energy Demand Forecasting

This notebook demonstrates Prophet-based forecasting for ERCOT energy demand using historical data and weather patterns.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:
# Load sample data (in production, this would come from BigQuery)
def generate_sample_data():
    """Generate sample ERCOT demand data"""
    start_date = datetime(2023, 1, 1)
    end_date = datetime(2024, 1, 1)
    
    dates = pd.date_range(start=start_date, end=end_date, freq='H')
    
    # Base demand with seasonal patterns
    base_demand = 50000
    
    # Seasonal components
    seasonal = 5000 * np.sin(2 * np.pi * np.arange(len(dates)) / (365.25 * 24))
    weekly = 2000 * np.sin(2 * np.pi * np.arange(len(dates)) / (7 * 24))
    daily = 3000 * np.sin(2 * np.pi * np.arange(len(dates)) / 24)
    
    # Temperature effect (simplified)
    temp_effect = 2000 * np.sin(2 * np.pi * np.arange(len(dates)) / (365.25 * 24) + np.pi/4)
    
    # Random noise
    noise = np.random.normal(0, 1000, len(dates))
    
    demand = base_demand + seasonal + weekly + daily + temp_effect + noise
    
    return pd.DataFrame({
        'timestamp': dates,
        'demand': demand,
        'temperature': 70 + 20 * np.sin(2 * np.pi * np.arange(len(dates)) / (365.25 * 24))
    })

df = generate_sample_data()
print(f"Generated {len(df)} hours of data")
df.head()


In [ ]:
# Explore the data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Time series plot
axes[0, 0].plot(df['timestamp'], df['demand'])
axes[0, 0].set_title('ERCOT Demand Over Time')
axes[0, 0].set_ylabel('Demand (MW)')

# Daily pattern
df['hour'] = df['timestamp'].dt.hour
daily_pattern = df.groupby('hour')['demand'].mean()
axes[0, 1].plot(daily_pattern.index, daily_pattern.values)
axes[0, 1].set_title('Average Daily Demand Pattern')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Average Demand (MW)')

# Weekly pattern
df['day_of_week'] = df['timestamp'].dt.dayofweek
weekly_pattern = df.groupby('day_of_week')['demand'].mean()
axes[1, 0].bar(weekly_pattern.index, weekly_pattern.values)
axes[1, 0].set_title('Average Weekly Demand Pattern')
axes[1, 0].set_xlabel('Day of Week')
axes[1, 0].set_ylabel('Average Demand (MW)')
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

# Temperature vs Demand
axes[1, 1].scatter(df['temperature'], df['demand'], alpha=0.1)
axes[1, 1].set_title('Temperature vs Demand')
axes[1, 1].set_xlabel('Temperature (°F)')
axes[1, 1].set_ylabel('Demand (MW)')

plt.tight_layout()
plt.show()
